# 11.1 — Aerospace Propulsion: A Crash Course

Propulsion is the science and engineering of generating thrust — the force that accelerates an aircraft against drag and gravity. Every flying vehicle, from a Wright Flyer to a hypersonic waverider, relies on Newton's third law: to move forward, something must be pushed backward. What distinguishes propulsion systems from one another is *what* is pushed backward, *how fast*, and *at what energy cost*.

This notebook provides a rigorous but accessible crash course covering the physical principles, major engine families, and key performance metrics that define aerospace propulsion. It is the foundation for the historical survey in **11.2** and the engineering detail in subsequent notebooks.

## 1. The Thrust Equation — Newton in a Tube

All air-breathing propulsion systems work by accelerating a mass flow of air (and combustion products) through a duct. Applying the momentum equation to a control volume around the engine:

$$\boxed{T = \dot{m}_0\,(V_e - V_0) + (p_e - p_0)\,A_e}$$

where

| Symbol | Meaning | Typical units |
|--------|---------|---------------|
| $T$ | Net thrust | N or lbf |
| $\dot{m}_0$ | Mass flow through engine | kg/s |
| $V_e$ | Exhaust velocity | m/s |
| $V_0$ | Aircraft (freestream) velocity | m/s |
| $p_e$ | Exhaust static pressure | Pa |
| $p_0$ | Ambient static pressure | Pa |
| $A_e$ | Nozzle exit area | m² |

The first term is **momentum thrust** — the rate of change of momentum of the air stream. The second is **pressure thrust**, which is zero for a perfectly expanded nozzle ($p_e = p_0$) and the dominant term in under-expanded rocket nozzles at altitude.

### The Rocket as a Special Case

A rocket carries its own oxidiser, so $\dot{m}_0$ includes both fuel and oxidiser, and $V_0$ does not reduce the effective jet velocity (there is no ingested air to consider). The rocket thrust equation simplifies to:

$$T_{\text{rocket}} = \dot{m}_e\,V_e + (p_e - p_0)\,A_e$$

### Installed vs. Uninstalled Thrust

The **gross thrust** $F_g = \dot{m}_0 V_e + (p_e - p_0)A_e$ does not account for the momentum of air already moving at $V_0$ that has been swallowed by the inlet. The **ram drag** $D_\text{ram} = \dot{m}_0 V_0$ must be subtracted to obtain the net thrust actually delivered to the airframe.

## 2. The Brayton Cycle — Thermodynamic Heart of the Gas Turbine

Gas turbine engines — turbojets, turbofans, turboprops — operate on the **ideal Brayton cycle**, which consists of four thermodynamic processes:

| Process | Path | Description |
|---------|------|-------------|
| 1 → 2 | Isentropic | Compression (compressor or inlet ram) |
| 2 → 3 | Isobaric | Heat addition (combustion at constant pressure) |
| 3 → 4 | Isentropic | Expansion (turbine + nozzle) |
| 4 → 1 | Isobaric | Heat rejection (exhaust to atmosphere) |

The **thermal efficiency** of the ideal Brayton cycle depends only on the overall pressure ratio $\pi_c$:

$$\eta_{\text{th}} = 1 - \frac{1}{\pi_c^{(\gamma-1)/\gamma}}$$

where $\gamma \approx 1.4$ for air. Higher pressure ratios — modern turbofans run at 40–60:1 — give higher thermal efficiency. Real engines are further constrained by:

- **Turbine entry temperature (TET)**: Limited by blade material and cooling technology. Modern engines reach 1,700–2,000 K — above the melting point of nickel superalloys — through sophisticated internal cooling.
- **Polytropic efficiency**: Real compression and expansion are not isentropic; losses reduce the ideal efficiency by 5–15%.
- **Combustion efficiency**: Typically 99.5% in modern engines.

In [ ]:
import matplotlib as mpl
mpl.rcParams.update({
    'figure.facecolor': 'none', 'axes.facecolor': 'none', 'savefig.facecolor': 'none', 'savefig.transparent': True,
    'axes.edgecolor':        '#888888',
    'axes.labelcolor':       '#cccccc',
    'text.color':            '#cccccc',
    'xtick.color':           '#cccccc',
    'ytick.color':           '#cccccc',
    'grid.color':            '#555555',
    'font.size': 11, 'axes.titlesize': 13, 'axes.labelsize': 11,
    'xtick.labelsize': 10, 'ytick.labelsize': 10,
    'legend.fontsize': 10
})
OKI = ['#0072B2','#D55E00','#009E73','#E69F00','#56B4E9','#CC79A7','#F0E442','#333333']

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# --- Left: Ideal Brayton Cycle on T-s diagram ---
ax = axes[0]
gamma = 1.4
# State points (s, T) — normalised
s1, T1 = 0.0, 1.0       # ambient
s2, T2 = 0.0, 2.8       # after isentropic compression (pi=20)
s3, T3 = 0.55, 6.5      # peak (TET)
s4, T4 = 0.55, 2.35     # after isentropic expansion

# Compression 1->2 (isentropic, vertical)
ax.annotate('', xy=(s2, T2), xytext=(s1, T1),
            arrowprops=dict(arrowstyle='->', color=OKI[0], lw=2))
ax.plot([s1, s2], [T1, T2], color=OKI[0], lw=2.5)

# Combustion 2->3 (isobaric)
s_cb = np.linspace(s2, s3, 100)
T_cb = np.linspace(T2, T3, 100)
ax.plot(s_cb, T_cb, color=OKI[1], lw=2.5)
ax.annotate('', xy=(s_cb[70], T_cb[70]), xytext=(s_cb[50], T_cb[50]),
            arrowprops=dict(arrowstyle='->', color=OKI[1], lw=2))

# Expansion 3->4 (isentropic, vertical)
ax.plot([s3, s4], [T3, T4], color=OKI[2], lw=2.5)
ax.annotate('', xy=(s4, T4+0.3), xytext=(s3, T3-0.3),
            arrowprops=dict(arrowstyle='->', color=OKI[2], lw=2))

# Exhaust 4->1 (isobaric heat rejection)
s_ex = np.linspace(s4, s1, 100)
T_ex = np.linspace(T4, T1, 100)
ax.plot(s_ex, T_ex, color=OKI[3], lw=2.5, linestyle='--')

# Fill net work area
s_fill = [s1, s2, s3, s4, s1]
T_fill = [T1, T2, T3, T4, T1]
ax.fill(s_fill, T_fill, alpha=0.12, color=OKI[0])

# State point labels
for (s, T, lbl, ha) in [(s1, T1, '1\n(Inlet)', 'right'), (s2, T2, '2\n(After compressor)', 'right'),
                          (s3, T3, '3\n(After combustor)', 'left'), (s4, T4, '4\n(After turbine)', 'left')]:
    ax.scatter(s, T, s=60, zorder=5, color='#333333')
    ax.annotate(lbl, (s, T), xytext=(8 if ha=='left' else -8, 0),
                textcoords='offset points', ha=ha, va='center', fontsize=9)

# Process labels
ax.text(-0.04, (T1+T2)/2, 'Compression\n(isentropic)', ha='right', va='center', fontsize=9, color=OKI[0])
ax.text((s2+s3)/2, T3+0.25, 'Combustion\n(isobaric)', ha='center', va='bottom', fontsize=9, color=OKI[1])
ax.text(s3+0.04, (T3+T4)/2, 'Expansion\n(isentropic)', ha='left', va='center', fontsize=9, color=OKI[2])
ax.text((s4+s1)/2, (T4+T1)/2-0.3, 'Exhaust\n(heat rejected)', ha='center', va='top', fontsize=9, color=OKI[3])
ax.text(0.2, 3.5, 'Net\nWork', ha='center', va='center', fontsize=10, color=OKI[0], alpha=0.8, style='italic')

ax.set_xlim(-0.15, 0.75)
ax.set_ylim(0.3, 7.2)
ax.set_xlabel('Specific Entropy, $s$ (normalised)')
ax.set_ylabel('Temperature, $T$ (normalised)')
ax.set_title('Ideal Brayton Cycle (T–s diagram)', fontweight='bold', pad=10)
ax.spines[['top','right']].set_visible(False)
ax.set_xticks([]); ax.set_yticks([])

# --- Right: Thermal efficiency vs pressure ratio ---
ax2 = axes[1]
pi_range = np.linspace(1, 60, 300)
for gamma_val, label, color in [(1.4, 'Ideal (γ=1.40)', OKI[0]),
                                  (1.33, 'Hot gas (γ=1.33)', OKI[1])]:
    eta = 1 - 1/pi_range**((gamma_val-1)/gamma_val)
    ax2.plot(pi_range, eta*100, lw=2.5, label=label, color=color)

# Annotate real engines
engines = [('RR Dart\n(1950)', 5.5, 1.4), ('JT8D\n(1960)', 16, 1.4),
           ('CFM56\n(1974)', 32, 1.4), ('GEnx\n(2006)', 45, 1.4), ('GE9X\n(2016)', 60, 1.4)]
for name, pi, gam in engines:
    eta_pt = (1 - 1/pi**((gam-1)/gam))*100
    ax2.scatter(pi, eta_pt, s=55, color=OKI[2], zorder=5)
    ax2.annotate(name, (pi, eta_pt), xytext=(0, 8), textcoords='offset points',
                 ha='center', fontsize=8, color='#333333')

ax2.set_xlabel('Overall Pressure Ratio (OPR)')
ax2.set_ylabel('Ideal Thermal Efficiency (%)')
ax2.set_title('Brayton Cycle: Efficiency vs Pressure Ratio', fontweight='bold', pad=10)
ax2.legend(frameon=False)
ax2.spines[['top','right']].set_visible(False)
ax2.set_xlim(0, 65)
ax2.set_ylim(0, 80)
ax2.grid(True, alpha=0.3, linestyle='--')

plt.tight_layout(pad=2)
plt.savefig('brayton_cycle.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Key Performance Metrics

### 3.1 Thrust-Specific Fuel Consumption (TSFC)

TSFC (or SFC) measures how efficiently an engine converts fuel energy into thrust:

$$\text{TSFC} = \frac{\dot{m}_f}{T}$$

Units: $\text{kg}/(\text{N}\cdot\text{s})$ or more practically $\text{lb}/(\text{lbf}\cdot\text{h})$. Lower is better. A modern turbofan at cruise has TSFC ≈ 0.50–0.55 lb/(lbf·h); a 1950s turbojet had ≈ 0.90–1.10.

### 3.2 Specific Impulse (Isp)

Specific impulse is the rocket-engineering analogue of TSFC — thrust per unit weight flow of propellant:

$$I_{sp} = \frac{T}{\dot{m}_f\,g_0} = \frac{1}{\text{TSFC}\cdot g_0}$$

Units: seconds. Useful for comparing air-breathers and rockets on a common basis. LH₂/LOX rockets reach ~450 s; kerosene turbofans at cruise effectively reach ~5,000–7,000 s (because they borrow oxygen from the atmosphere).

### 3.3 Propulsive Efficiency

Not all jet kinetic energy does useful work — some accelerates the exhaust beyond what is needed. Propulsive efficiency is:

$$\eta_p = \frac{T\,V_0}{\text{Kinetic power of jet}} = \frac{2}{1 + V_e/V_0}$$

Key insight: **propulsive efficiency peaks when the exhaust velocity equals the flight velocity ($V_e \to V_0$)**. A very high jet velocity wastes energy in a fast-moving exhaust. This is why turbofans with large bypass ratios (which slow the jet by moving more air) are more efficient at subsonic cruise than turbojets.

### 3.4 Overall Efficiency

$$\eta_0 = \eta_{\text{th}} \times \eta_p = \frac{T\,V_0}{\dot{m}_f\,Q_R}$$

where $Q_R \approx 43\,\text{MJ/kg}$ is the lower heating value of kerosene. Modern turbofans achieve $\eta_0 \approx 35\text{–}40\%$ at cruise.

## 4. Reciprocating (Piston) Engines

The piston engine — which dominated aviation from 1903 to the late 1940s — operates on the **Otto cycle** (for spark-ignition) or **Diesel cycle** (for compression-ignition). In aviation the four-stroke spark-ignition engine is almost universal.

### Operating Principle

1. **Induction**: Piston moves down, drawing fuel-air mixture into the cylinder.
2. **Compression**: Piston moves up, compressing the mixture (compression ratio typically 6:1–8:1 for aviation, limited by detonation).
3. **Power**: Spark ignites compressed mixture; expanding gases push piston down.
4. **Exhaust**: Piston moves up, expelling combustion gases.

The crankshaft converts reciprocating motion to rotation, driving a propeller either directly or through a reduction gearbox.

### Key Characteristics

- **Power-to-weight ratio**: 0.5–1.5 kW/kg (much lower than turbines)
- **BSFC**: Typically 0.30–0.45 kg/(kW·h) — very efficient at low power settings
- **Altitude sensitivity**: Power falls rapidly with altitude (air density) unless supercharged or turbocharged
- **Speed limit**: Propeller tip speeds limit practical aircraft speeds to ~750 km/h
- **Best suited to**: Light aircraft, general aviation, UAVs requiring long endurance at low speed

### The Supercharger and Turbocharger

To restore sea-level power at altitude, a mechanically driven **supercharger** (Merlin, Griffon) or exhaust-driven **turbocharger** (most modern GA engines) compresses the intake air before it enters the cylinders. The Rolls-Royce Merlin used a two-speed, two-stage supercharger that allowed it to maintain power to 7,600 m.

## 5. The Turbojet — Pure Jet Propulsion

The turbojet is the simplest continuous-combustion gas turbine. Air is ingested, compressed by a multi-stage axial compressor, mixed with fuel and burned in the combustion chamber, then expanded through a turbine (which extracts just enough energy to drive the compressor) and exhausted at high velocity through a nozzle.

### Component Flow Path

$$\text{Inlet} \to \text{Compressor} \to \text{Combustor} \to \text{Turbine} \to \text{Nozzle}$$

The energy balance requires:
$$W_{\text{turbine}} = W_{\text{compressor}} + \text{accessory loads}$$

All remaining energy appears as kinetic energy in the exhaust jet.

### Strengths and Weaknesses

| Attribute | Value |
|-----------|-------|
| Thrust-to-weight ratio | 4–6:1 (uninstalled) |
| TSFC at cruise | 0.85–1.10 lb/(lbf·h) |
| Optimal Mach range | 2.0–3.0 |
| Propulsive efficiency at M=0.85 | ~35% |

The high jet velocity that gives turbojets their supersonic capability is precisely what makes them inefficient at subsonic cruise — most of the exhaust's kinetic energy is wasted. This insight drove the development of the turbofan.

### The Afterburner

A **reheat (afterburner)** section aft of the turbine injects additional fuel into the hot, oxygen-rich turbine exhaust. Since there is no mechanical work to extract, the additional heat appears directly as increased jet velocity. Afterburning can increase thrust by 50–70% at the cost of roughly tripling fuel consumption — so it is used only for short bursts of acceleration or combat manoeuvring.

## 6. The Turbofan — Efficiency Through Bypass

The turbofan adds a large fan ahead of the core compressor, driven by an additional turbine stage. The fan moves a much larger mass of air — the **bypass stream** — around the core, accelerating it gently to produce thrust without combustion.

$$\text{Bypass Ratio (BPR)} = \frac{\dot{m}_{\text{bypass}}}{\dot{m}_{\text{core}}}$$

Modern civil turbofans have BPR of 8–13 (LEAP, GEnx, Trent XWB); the GE9X reaches ~16. The result is a larger mass of air accelerated through a smaller $\Delta V$, which — per the propulsive efficiency equation — dramatically reduces wasted kinetic energy.

```{figure} https://upload.wikimedia.org/wikipedia/commons/thumb/2/20/Simplified_turbofan_engine_cutaway.svg/1280px-Simplified_turbofan_engine_cutaway.svg.png
:name: fig-turbofan-cutaway
:width: 90%
:align: center
Cutaway schematic of a high-bypass turbofan showing the fan, bypass duct, core compressor stages, combustor, turbine stages, and exhaust nozzles. *Wikimedia Commons, CC BY-SA.*
```

### Geared Turbofan (GTF)

Increasing the BPR requires a larger, slower fan and a faster, smaller core turbine. A **reduction gearbox** between them (as in the Pratt & Whitney PW1000G series) allows each to operate at its optimal speed, enabling BPR > 12 without penalising core efficiency. The GTF delivers 15–20% lower fuel burn than the engines it replaces.

### Military Low-Bypass Turbofans

Fighter engines use BPR of 0.2–1.0 — a compromise between the efficiency of high bypass and the compact frontal area needed for supersonic flight. The dry TSFC penalty is accepted because the aircraft spends most of its time at subsonic cruise, and the small fan diameter keeps supersonic drag manageable.

```{figure} https://upload.wikimedia.org/wikipedia/commons/thumb/5/5c/Pratt_%26_Whitney_F100-PW-220_turbofan_engine.jpg/1280px-Pratt_%26_Whitney_F100-PW-220_turbofan_engine.jpg
:name: fig-f100-engine
:width: 80%
:align: center
Pratt & Whitney F100-PW-220 low-bypass turbofan (BPR ≈ 0.36), the engine that powers the F-15 Eagle and F-16 Fighting Falcon. With afterburner it produces ~130 kN thrust from a 1,678 kg engine — a thrust-to-weight ratio exceeding 8:1. *U.S. Air Force photo, public domain.*
```

## 7. Turboprops and Turboshafts — Power Through a Shaft

In a **turboprop**, the gas turbine extracts almost all of the available energy from the gas stream through an extended turbine section, delivering it as shaft power to a conventional propeller through a reduction gearbox. A small residual jet thrust (typically 10–20% of total) comes from the exhaust nozzle.

The turboprop inherits the propeller's high propulsive efficiency at low speeds. Because the propeller moves a very large mass of air at low velocity, the propulsive efficiency can reach 80–85% at 400–600 km/h — far exceeding that of any jet at the same speed. The penalty is that propulsive efficiency collapses above ~650 km/h as blade tips approach supersonic speeds.

$$\eta_{\text{total, turboprop}} = \eta_{\text{mech}} \times \eta_{\text{prop}} \times \eta_{\text{th}}$$

**Turboshaft** engines are identical in principle but deliver all their power as shaft output, with minimal exhaust jet thrust. They power helicopters, hovercraft, ships, and gas-turbine locomotives. The turboshaft must accommodate widely varying power demands as the rotor changes collective pitch.

### Key Specifications Compared

| Parameter | Turboprop | Turboshaft |
|-----------|-----------|------------|
| Power output | 500 – 5,000 shp | 500 – 6,000 shp |
| Equivalent TSFC | 0.40 – 0.55 kg/(kW·h) | 0.25 – 0.45 kg/(kW·h) |
| T/W ratio | 3–5:1 | 4–7:1 |
| Best cruise speed | 400 – 650 km/h | N/A (hover) |

## 8. Ramjets and Scramjets — Compression Without Machinery

### The Ramjet

At supersonic flight speeds, ram compression alone can achieve pressure ratios sufficient to sustain combustion — no rotating compressor is needed. A ramjet is simply an inlet, a combustion chamber, and a nozzle.

- **Minimum operating Mach**: ~M 2.5 (needs ram pressure to sustain combustion)
- **Optimal Mach**: 3–5
- **Cannot self-start**: Must be accelerated to operating speed by a booster
- **Applications**: Missiles (ASRAAM, BrahMos), hypersonic test vehicles

The inlet **decelerates** the supersonic flow to subsonic speeds before the combustor (with strong shocks), heating the air to 600–900 K — sometimes requiring heat-resistant combustor linings and careful fuel management.

### The Scramjet — Supersonic Combustion

Above M 5–6, decelerating the flow to subsonic speeds for combustion generates so much heat that the air chemically dissociates, destroying efficiency. The **Scramjet** (Supersonic Combustion Ramjet) maintains supersonic flow *through* the combustor — fuel must mix and burn in milliseconds in a flow moving at 1,500–3,000 m/s.

- **Operating range**: M 5 – ~M 25 (theoretical)
- **Technology readiness**: TRL 4–6 (demonstrated in short bursts: X-43A at M 9.8, X-51A at M 5.1)
- **Challenge**: Combustion stability, thermal management, structural loads in high-dynamic-pressure flight

No scramjet-powered aircraft has yet demonstrated sustained, controlled flight; all operational applications remain in the missile and experimental vehicle domain.

## 9. Rocket Propulsion — Independence from the Atmosphere

Rockets carry both fuel and oxidiser, making them independent of atmospheric oxygen. This enables operation from sea level to orbit and beyond, but at the cost of carrying an enormous mass of propellant.

### The Tsiolkovsky Rocket Equation

The fundamental constraint on rocket performance is the **rocket equation**:

$$\Delta V = I_{sp}\,g_0\,\ln\!\left(\frac{m_0}{m_f}\right)$$

where $m_0/m_f$ is the mass ratio (initial to final, i.e., full to empty of propellant). To reach low Earth orbit ($\Delta V \approx 9.4\,\text{km/s}$) with $I_{sp} \approx 450\,\text{s}$ (LH₂/LOX), the mass ratio must be ~8.3 — meaning 88% of launch mass is propellant.

### Propellant Combinations

| Combination | $I_{sp}$ (vac, s) | Notes |
|-------------|-------------------|-------|
| RP-1 / LOX | 350 | Dense, storable; Saturn V S-IC, Falcon 9 |
| LH₂ / LOX | 450 | Highest Isp; cryogenic handling challenge |
| N₂O₄ / UDMH | 340 | Hypergolic (self-igniting); storable |
| Solid (APCP) | 250–280 | Simple, reliable; no throttle control |
| CH₄ / LOX | 380 | Reusability-friendly; SpaceX Raptor |

### Aviation Relevance

Rockets are used in aviation for: missile propulsion, JATO (jet-assisted take-off), the Bell X-1 (first supersonic aircraft), the X-15 (first aircraft to reach space), SpaceShipOne/Two, and rocket-assisted fighter escape systems.

## 10. Electric Propulsion — Motors, Batteries, and Fuel Cells

Electric propulsion converts electrical energy into shaft power driving a propeller or fan. The advantages over combustion — no local emissions, low noise, high motor efficiency (95–98%), instant torque — are offset by the low energy density of current batteries.

### Energy Density Comparison

| Source | Specific energy (Wh/kg) |
|--------|------------------------|
| Jet-A kerosene | ~12,000 |
| Hydrogen (liquid) | ~33,000 |
| Li-ion battery (cell, 2024) | 250–300 |
| Li-S battery (emerging) | 400–600 |
| Solid-state battery (2030s target) | 500–700 |

The 40–50× energy density deficit of batteries versus kerosene means that full electrification of commercial aviation is limited to short ranges (< 500 km, < 19 seats) for the foreseeable future.

### Hybrid-Electric Architecture

A **serial hybrid** uses a gas turbine generator to power electric motors — decoupling shaft speed from flight condition allows the turbine to run at its most efficient point. A **parallel hybrid** supplements gas turbine thrust with electric boost during take-off and climb (when fuel burn per unit thrust is highest), then recharges during cruise.

### Hydrogen Fuel Cells

Hydrogen fuel cells (PEM type) convert H₂ + O₂ electrochemically to water, producing electricity at ~60% efficiency. Combined with electric motors, the system emits only water vapour. The challenges are hydrogen storage (cryogenic liquid at -253°C, or high-pressure 700 bar) and the absence of hydrogen production/distribution infrastructure at airports.

```{figure} https://upload.wikimedia.org/wikipedia/commons/thumb/1/1c/Piloted%2C_Electric_Propulsion-Powered_Experimental_Aircraft_Underway_%28ED15-0373-32%29.jpg/1280px-Piloted%2C_Electric_Propulsion-Powered_Experimental_Aircraft_Underway_%28ED15-0373-32%29.jpg
:name: fig-nasa-electric
:width: 80%
:align: center
NASA's X-57 Maxwell experimental aircraft with distributed electric propulsion — 14 small electric motors on the leading edge for low-speed lift augmentation, plus two wingtip cruise motors. *NASA photo, public domain.*
```

In [ ]:
import matplotlib as mpl
mpl.rcParams.update({
    'figure.facecolor': 'none', 'axes.facecolor': 'none', 'savefig.facecolor': 'none', 'savefig.transparent': True,
    'axes.edgecolor':        '#888888',
    'axes.labelcolor':       '#cccccc',
    'text.color':            '#cccccc',
    'xtick.color':           '#cccccc',
    'ytick.color':           '#cccccc',
    'grid.color':            '#555555',
    'font.size': 11, 'axes.titlesize': 13, 'axes.labelsize': 11,
    'xtick.labelsize': 10, 'ytick.labelsize': 10,
    'legend.fontsize': 10
})
OKI = ['#0072B2','#D55E00','#009E73','#E69F00','#56B4E9','#CC79A7','#F0E442','#333333']

import numpy as np
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# --- Left: Propulsive efficiency vs Mach for engine types ---
ax = axes[0]
mach = np.linspace(0.05, 4.0, 400)

def prop_eff(Ve_V0):
    """Propulsive efficiency = 2/(1 + Ve/V0)"""
    return 2 / (1 + Ve_V0)

# Propeller/piston: Ve/V0 ≈ 1.03–1.06 (very small added velocity)
mach_prop = np.linspace(0.05, 0.75, 200)
eta_prop = prop_eff(1.04) * np.ones_like(mach_prop) * (1 - 0.3*(mach_prop/0.75)**3)
ax.fill_between(mach_prop, eta_prop*0.88, eta_prop*1.02, alpha=0.25, color=OKI[0])
ax.plot(mach_prop, eta_prop*0.94, lw=2.5, color=OKI[0], label='Propeller (piston/turboprop)')

# Turbofan high BPR: η_p peaks ~0.7 around M0.85
mach_tf = np.linspace(0.2, 1.05, 200)
eta_tf = 0.71 * np.exp(-0.5*((mach_tf - 0.85)/0.3)**2) + 0.45*(1 - np.exp(-3*mach_tf))
eta_tf = np.clip(eta_tf, 0, 0.76)
ax.fill_between(mach_tf, eta_tf*0.9, eta_tf*1.05, alpha=0.2, color=OKI[2])
ax.plot(mach_tf, eta_tf, lw=2.5, color=OKI[2], label='High-bypass turbofan (BPR 10–13)')

# Turbojet: lower η_p at subsonic, better at supersonic
mach_tj = np.linspace(0.3, 3.5, 300)
eta_tj = 2*mach_tj / (mach_tj + 2.2)
eta_tj = np.clip(eta_tj, 0, 0.68)
ax.fill_between(mach_tj, eta_tj*0.88, eta_tj*1.0, alpha=0.15, color=OKI[1])
ax.plot(mach_tj, eta_tj*0.94, lw=2.5, color=OKI[1], label='Turbojet')

# Ramjet
mach_rj = np.linspace(2.0, 4.0, 100)
eta_rj = 2*mach_rj / (mach_rj + 1.5) * 0.55
ax.plot(mach_rj, eta_rj, lw=2.5, color=OKI[3], label='Ramjet', linestyle='--')

ax.set_xlabel('Flight Mach Number')
ax.set_ylabel('Propulsive Efficiency, $\\eta_p$')
ax.set_title('Propulsive Efficiency vs Mach Number', fontweight='bold')
ax.legend(frameon=False, loc='lower right', fontsize=9)
ax.spines[['top','right']].set_visible(False)
ax.set_xlim(0, 4.2)
ax.set_ylim(0, 1.0)
ax.yaxis.set_major_formatter(mpl.ticker.PercentFormatter(xmax=1))
ax.grid(True, alpha=0.3, linestyle='--')

# Annotate optimal zones
ax.axvspan(0, 0.75, alpha=0.04, color=OKI[0])
ax.axvspan(0.7, 1.05, alpha=0.04, color=OKI[2])
ax.axvspan(2.0, 4.2, alpha=0.04, color=OKI[3])
ax.text(0.35, 0.92, 'Prop\nregion', ha='center', va='top', fontsize=8, color=OKI[0])
ax.text(0.87, 0.92, 'Fan\nregion', ha='center', va='top', fontsize=8, color=OKI[2])
ax.text(3.1, 0.92, 'Ram\nregion', ha='center', va='top', fontsize=8, color=OKI[3])

# --- Right: TSFC comparison bar chart ---
ax2 = axes[1]
engines = ['Piston\n(cruise)', 'Turboprop\n(cruise)', 'High-BPR\nTurbofan', 'Low-BPR\nTurbofan', 'Turbojet\n(dry)', 'Turbojet\n(afterburner)', 'Ramjet']
# TSFC in lb/(lbf·h) — typical cruise values
tsfc_lo = [0.36, 0.42, 0.48, 0.62, 0.85, 2.0, 2.5]
tsfc_hi = [0.50, 0.58, 0.58, 0.80, 1.10, 2.8, 3.5]
tsfc_mid = [(lo+hi)/2 for lo, hi in zip(tsfc_lo, tsfc_hi)]
err = [(mid-lo, hi-mid) for mid, lo, hi in zip(tsfc_mid, tsfc_lo, tsfc_hi)]
err_lo = [e[0] for e in err]
err_hi = [e[1] for e in err]
colors = [OKI[0], OKI[2], OKI[2], OKI[1], OKI[1], OKI[3], OKI[3]]

bars = ax2.barh(engines, tsfc_mid, xerr=[err_lo, err_hi], capsize=4,
                color=colors, alpha=0.85, edgecolor='white', linewidth=0.5,
                error_kw={'elinewidth': 1.5, 'ecolor': '#555'})

ax2.set_xlabel('TSFC  [lb / (lbf · h)]   ← lower is better')
ax2.set_title('Cruise TSFC by Engine Type', fontweight='bold')
ax2.spines[['top','right']].set_visible(False)
ax2.axvline(1.0, color='#aaa', linestyle=':', lw=1.5)
ax2.set_xlim(0, 4.0)
ax2.grid(True, axis='x', alpha=0.3, linestyle='--')

# Legend
from matplotlib.patches import Patch
legend_patches = [
    Patch(facecolor=OKI[0], label='Shaft / propeller engines'),
    Patch(facecolor=OKI[2], label='Turbofan'),
    Patch(facecolor=OKI[1], label='Turbojet'),
    Patch(facecolor=OKI[3], label='Ramjet'),
]
ax2.legend(handles=legend_patches, frameon=False, fontsize=9, loc='lower right')

plt.tight_layout(pad=2)
plt.savefig('tsfc_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. The Modern High-Bypass Turbofan in Detail

The state of the art in subsonic propulsion is represented by engines like the GE90, GEnx, GE9X, Rolls-Royce Trent XWB, and CFM LEAP. These engines embody six decades of compressor, turbine, and materials research.

```{figure} https://upload.wikimedia.org/wikipedia/commons/thumb/6/66/General_Electric_GE90-115B.jpg/1280px-General_Electric_GE90-115B.jpg
:name: fig-ge90
:width: 85%
:align: center
The General Electric GE90-115B — the world's most powerful production turbofan at 115,540 lbf (514 kN) thrust. Its 3.25 m fan diameter and BPR of ~9 power the Boeing 777-300ER. The carbon-fibre composite fan blades were among the first used on a commercial engine. *Wikimedia Commons, CC BY-SA.*
```

### Trent XWB — Ultra-High Efficiency

```{figure} https://upload.wikimedia.org/wikipedia/commons/thumb/2/2a/A350_Triebwerk_TRENT_XWB_%2841066981724%29.jpg/1280px-A350_Triebwerk_TRENT_XWB_%2841066981724%29.jpg
:name: fig-trent-xwb
:width: 85%
:align: center
The Rolls-Royce Trent XWB on an Airbus A350 XWB. With a BPR of 9.3 and OPR of 52, it achieves the lowest TSFC of any large commercial turbofan at its entry into service. Its three-shaft architecture (fan, intermediate compressor, and high-pressure core each on separate concentric shafts) allows each spool to rotate at its optimal speed. *Wikimedia Commons, CC BY-SA.*
```

### Key Technological Features of Modern Turbofans

- **Carbon-fibre composite fan blades**: Wide-chord, swept blades reducing weight and improving aerodynamic efficiency. The GE9X fan blade is the world's first commercial-engine blade made entirely from ceramic matrix composite (CMC) at the leading edge.
- **Single-crystal turbine blades**: Cast from a single metallic crystal to eliminate grain boundaries, allowing operating temperatures 200–300°C above polycrystalline nickel.
- **Thermal barrier coatings (TBCs)**: Ceramic coatings on turbine blades that insulate the metal from gas stream temperatures exceeding 2,000 K.
- **3D-printed components**: Fuel nozzles (GE LEAP), heat exchangers, and brackets produced by selective laser sintering — fewer parts, lighter assemblies, better aerodynamic performance.
- **FADEC** (Full Authority Digital Engine Control): Continuously optimises fuel flow, compressor variable geometry, and bleed valves to maximise efficiency across every phase of flight.

## 12. Matching Propulsion to Mission

Propulsion selection is a design-driver decision made early in the conceptual design phase. The primary axes of selection are **speed regime** and **altitude**, since these define the thermodynamic environment the engine operates in. Secondary factors include required thrust, fuel availability, maintenance environment, noise constraints, and development risk.

### Decision Framework

| Mission type | Mach range | Recommended propulsion | Rationale |
|---|---|---|---|
| Long endurance UAV | 0.05–0.20 | Piston / fuel cell | Minimum SFC, low cost |
| Regional turboprop | 0.40–0.65 | Turboprop | Prop efficiency + turbine reliability |
| Short-haul airliner | 0.78–0.82 | High-BPR turbofan | Cruise efficiency, noise |
| Long-haul airliner | 0.82–0.86 | Ultra-high-BPR turbofan | Block fuel, TSFC critical |
| Fighter (multirole) | 0.9–1.8 | Low-BPR augmented turbofan | T/W ratio, supersonic dash |
| Interceptor / Mach 2+ | 1.5–3.0 | Turbojet with afterburner | High Mach capability |
| Cruise missile / target drone | 2.5–4.5 | Ramjet | Simplicity, speed |
| Launch vehicle (1st stage) | 0–Mach 8 | LH₂/LOX or RP-1/LOX rocket | No atmospheric oxygen needed |
| Hypersonic vehicle | 5–25 | Scramjet + rocket | No rotating machinery |

The selection process involves iteration between the propulsion constraints and the aircraft sizing — changes in engine bypass ratio affect nacelle drag; changes in thrust loading affect wing area and fuel mass, which feeds back to the engine thrust requirement.

In [ ]:
import matplotlib as mpl
mpl.rcParams.update({
    'figure.facecolor': 'none', 'axes.facecolor': 'none', 'savefig.facecolor': 'none', 'savefig.transparent': True,
    'axes.edgecolor':        '#888888',
    'axes.labelcolor':       '#cccccc',
    'text.color':            '#cccccc',
    'xtick.color':           '#cccccc',
    'ytick.color':           '#cccccc',
    'grid.color':            '#555555',
    'font.size': 10, 'axes.titlesize': 12
})
OKI = ['#0072B2','#D55E00','#009E73','#E69F00','#56B4E9','#CC79A7','#F0E442','#333333']

import numpy as np
import matplotlib.pyplot as plt

# Spider/radar chart comparing engine types across attributes
attrs = ['Fuel\nEfficiency', 'Power-to-\nWeight', 'Speed\nCapability', 'Simplicity /\nReliability',
         'Low-Speed\nEfficiency', 'Altitude\nCapability']
N = len(attrs)
angles = np.linspace(0, 2*np.pi, N, endpoint=False).tolist()
angles += angles[:1]

engine_data = [
    ('Piston engine',       [0.85, 0.30, 0.10, 0.75, 0.95, 0.30], OKI[0]),
    ('Turboprop',           [0.75, 0.55, 0.25, 0.70, 0.80, 0.65], OKI[2]),
    ('High-BPR Turbofan',   [0.90, 0.60, 0.55, 0.80, 0.30, 0.90], OKI[1]),
    ('Low-BPR Turbofan',    [0.55, 0.90, 0.80, 0.75, 0.20, 0.95], OKI[3]),
    ('Ramjet',              [0.45, 0.95, 0.90, 0.85, 0.00, 0.90], OKI[4]),
]

fig, ax = plt.subplots(figsize=(8, 7), subplot_kw=dict(polar=True))
for name, vals, color in engine_data:
    vals_plot = vals + vals[:1]
    ax.plot(angles, vals_plot, 'o-', lw=2, color=color, label=name, markersize=4)
    ax.fill(angles, vals_plot, alpha=0.08, color=color)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(attrs, fontsize=10)
ax.set_ylim(0, 1)
ax.set_yticks([0.25, 0.50, 0.75, 1.0])
ax.set_yticklabels(['25%', '50%', '75%', '100%'], fontsize=8)
ax.set_title('Engine Type Capability Comparison\n(relative scores, not absolute)', 
             fontweight='bold', pad=25)
ax.legend(loc='upper right', bbox_to_anchor=(1.38, 1.15), frameon=False, fontsize=9)
ax.grid(True, alpha=0.4)

plt.tight_layout()
plt.savefig('engine_radar.png', dpi=150, bbox_inches='tight')
plt.show()

## 13. Bibliography and Further Reading

```{bibliography}
:filter: False
:list: enumerated
```

**Core references for this notebook:**

- Mattingly, J. D., Heiser, W. H., & Pratt, D. T. (2002). *Aircraft Engine Design* (2nd ed.). AIAA Education Series.
- Cumpsty, N. A. (2003). *Jet Propulsion: A Simple Guide to the Aerodynamics and Thermodynamic Design and Performance of Jet Engines* (2nd ed.). Cambridge University Press.
- Hill, P. G., & Peterson, C. R. (1992). *Mechanics and Thermodynamics of Propulsion* (2nd ed.). Addison-Wesley.
- Farokhi, S. (2014). *Aircraft Propulsion* (2nd ed.). Wiley.
- Kerrebrock, J. L. (1992). *Aircraft Engines and Gas Turbines* (2nd ed.). MIT Press.
- Sutton, G. P., & Biblarz, O. (2017). *Rocket Propulsion Elements* (9th ed.). Wiley.
- Walsh, P. P., & Fletcher, P. (2004). *Gas Turbine Performance* (2nd ed.). Blackwell Science.